# Vietnamese Question-Type Classification
Fine-tune an encoder-only transformer to classify Vietnamese questions into 4 labels.
Assumptions: CSVs in `DATA_PATH` include `question` and `question_type` columns; GPU is optional.

## Environment Setup

In [ ]:
# %pip uninstall -y torch torchvision torchaudio
%pip install -q torch transformers datasets scikit-learn pandas matplotlib

In [ ]:
import os
import random
import json
import numpy as np
import pandas as pd
import torch, sys

print("torch", torch.__version__)
print("cuda", torch.version.cuda)
print("gpu", torch.cuda.get_device_name(0))
print("cap", torch.cuda.get_device_capability(0))

from datasets import load_dataset, ClassLabel, concatenate_datasets
from IPython.display import display
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
 )

import matplotlib.pyplot as plt

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Config

In [ ]:
MODEL_NAME = "vinai/phobert-base-v2"
# MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
NUM_CYCLES = 1
LR_SCHEDULER = "cosine_with_restarts"
LOGGING_STEPS = 50
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.0
DATA_PATH = "/kaggle/input/datasets/thuanhuynh2004/data-qa-final-merge"
SAVE_PATH = "/kaggle/working/models/{}_question_type_classifier".format(MODEL_NAME.split("/")[-1])
RESULTS_DIR = os.path.join(SAVE_PATH, "results")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_PER_LABEL = {
    "VERIFICATION": None,
    "FACTOID": 5000,
    "SUMMARY": 5000,
    "COMPARISON": 5000,
}

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

TEXT_COL = "question"
LABEL_COL = "question_type"
GRAD_CLIP = 1.0
PLOT_FIGSIZE = (10, 4)

if DEVICE == "cuda":
    try:
        torch.zeros(1, device="cuda")
    except Exception:
        DEVICE = "cpu"

## Data Loading & Inspection

In [ ]:
csv_files = [os.path.join(DATA_PATH, "train.csv")]  # training split only
csv_files = sorted(csv_files)
dataset = load_dataset("csv", data_files=csv_files, split="train")

missing = {TEXT_COL, LABEL_COL} - set(dataset.column_names)
if missing:
    raise ValueError(f"Missing columns: {missing}")

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

dataset = dataset.filter(lambda x: x[TEXT_COL] is not None and x[LABEL_COL] is not None)

label_names = sorted(set(dataset[LABEL_COL]))
dataset = dataset.cast_column(LABEL_COL, ClassLabel(names=label_names))
label2id = {name: idx for idx, name in enumerate(label_names)}
id2label = {idx: name for idx, name in enumerate(label_names)}
NUM_LABELS = len(label_names)

print("files:", [os.path.basename(f) for f in csv_files])
print("rows:", dataset.num_rows)
display(dataset.select(range(min(3, dataset.num_rows))).to_pandas()[[TEXT_COL, LABEL_COL]])

label_counts = pd.Series(dataset[LABEL_COL]).value_counts().rename(index=id2label)
display(label_counts)
label_counts.to_csv(os.path.join(RESULTS_DIR, "label_distribution_full.csv"), header=True)

ax = label_counts.sort_index().plot(kind="bar", figsize=(8, 4), title="Label distribution (full)")
ax.set_xlabel("Label")
ax.set_ylabel("Count")
fig = plt.gcf()
fig.savefig(os.path.join(RESULTS_DIR, "label_distribution_full.png"), dpi=150, bbox_inches="tight")
plt.show()

sampled_splits = []
actual_counts = {}
for label_name, label_id in label2id.items():
    target = TARGET_PER_LABEL.get(label_name, 5000)
    label_ds = dataset.filter(lambda x, label_id=label_id: x[LABEL_COL] == label_id)
    if target is not None and label_ds.num_rows > target:
        label_ds = label_ds.shuffle(seed=SEED).select(range(target))
    actual_counts[label_name] = label_ds.num_rows
    sampled_splits.append(label_ds)

dataset_sample = concatenate_datasets(sampled_splits).shuffle(seed=SEED)

sample_label_counts = pd.Series(dataset_sample[LABEL_COL]).value_counts().rename(index=id2label)
sample_label_counts.to_csv(os.path.join(RESULTS_DIR, "label_distribution_sample.csv"), header=True)
ax = sample_label_counts.sort_index().plot(kind="bar", figsize=(8, 4), title="Label distribution (sample)")
ax.set_xlabel("Label")
ax.set_ylabel("Count")
fig = plt.gcf()
fig.savefig(os.path.join(RESULTS_DIR, "label_distribution_sample.png"), dpi=150, bbox_inches="tight")
plt.show()

split_1 = dataset_sample.train_test_split(
    test_size=(1.0 - TRAIN_RATIO),
    stratify_by_column=LABEL_COL,
    seed=SEED,
 )
train_ds = split_1["train"]
temp_ds = split_1["test"]

val_ratio_adjusted = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
split_2 = temp_ds.train_test_split(
    test_size=(1.0 - val_ratio_adjusted),
    stratify_by_column=LABEL_COL,
    seed=SEED,
 )
val_ds = split_2["train"]
test_ds = split_2["test"]

def add_label_id(batch):
    batch["labels"] = batch[LABEL_COL]
    return batch

train_ds = train_ds.map(add_label_id, batched=True)
val_ds = val_ds.map(add_label_id, batched=True)
test_ds = test_ds.map(add_label_id, batched=True)

split_sizes = {
    "full_rows": dataset.num_rows,
    "sample_rows": dataset_sample.num_rows,
    "per_label_counts": actual_counts,
    "train_rows": train_ds.num_rows,
    "val_rows": val_ds.num_rows,
    "test_rows": test_ds.num_rows,
}
run_config = {
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_RATIO,
    "lr_scheduler": LR_SCHEDULER,
    "num_cycles": NUM_CYCLES,
    "seed": SEED,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
    "target_per_label": TARGET_PER_LABEL,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
}
metadata = {
    "split_sizes": split_sizes,
    "label2id": label2id,
    "id2label": id2label,
    "config": run_config,
}
with open(os.path.join(RESULTS_DIR, "run_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("sample size:", dataset_sample.num_rows)
print("per-label counts:", actual_counts)
print("train/val/test:", train_ds.num_rows, val_ds.num_rows, test_ds.num_rows)

## Class Weight Computation

In [ ]:
classes = np.arange(NUM_LABELS)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(train_ds["labels"]),
)
class_weights = torch.tensor(weights, dtype=torch.float32)
class_weights = class_weights.to(DEVICE)
print("class_weights:", class_weights)

## Preprocessing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)

train_hf = train_ds.map(tokenize_batch, batched=True)
val_hf = val_ds.map(tokenize_batch, batched=True)
test_hf = test_ds.map(tokenize_batch, batched=True)

sample_text = test_ds[0][TEXT_COL]
sample_gold_label = test_ds[0][LABEL_COL]

columns = ["input_ids", "attention_mask", "labels"]
train_hf.set_format(type="torch", columns=columns)
val_hf.set_format(type="torch", columns=columns)
test_hf.set_format(type="torch", columns=columns)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# token distribution count
def token_length_stats(dataset, tokenizer, text_col, percentiles=[50, 75, 90, 95, 99]):
    lengths = [len(tokenizer.encode(t)) for t in dataset[text_col]]
    s = pd.Series(lengths)
    stats = s.describe(percentiles=[p/100 for p in percentiles])
    print(stats.to_string())
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    s.hist(bins=50, ax=axes[0])
    axes[0].set_title("Token length distribution")
    axes[0].set_xlabel("Tokens")
    axes[0].axvline(MAX_LEN, color="red", linestyle="--", label=f"MAX_LEN={MAX_LEN}")
    axes[0].legend()
    
    s.plot(kind="box", ax=axes[1])
    axes[1].set_title("Token length boxplot")
    plt.tight_layout()
    plt.show()
    
    over = (s > MAX_LEN).sum()
    print(f"\nsequences exceeding MAX_LEN ({MAX_LEN}): {over} ({over/len(s)*100:.1f}%)")

token_length_stats(dataset_sample, tokenizer, TEXT_COL)

## Model Definition

In [ ]:
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    label2id=label2id,
    id2label=id2label,
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
model.to(DEVICE)

param_count = sum(p.numel() for p in model.parameters())
print(f"parameters: {param_count:,}")

## Training Loop

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=class_weights
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir=SAVE_PATH,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    lr_scheduler_kwargs={"num_cycles": NUM_CYCLES},
    logging_steps=LOGGING_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    report_to="none",
    max_grad_norm=GRAD_CLIP,
 )

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
 )

trainer.train()

log_history = trainer.state.log_history
train_points = [(item["epoch"], item["loss"]) for item in log_history if "loss" in item and "epoch" in item and "eval_loss" not in item]
eval_points = [(item["epoch"], item["eval_f1_macro"]) for item in log_history if "eval_f1_macro" in item and "epoch" in item]

fig, axes = plt.subplots(1, 2, figsize=PLOT_FIGSIZE)
if train_points:
    axes[0].plot([p[0] for p in train_points], [p[1] for p in train_points], marker="o")
axes[0].set_title("Train loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

if eval_points:
    axes[1].plot([p[0] for p in eval_points], [p[1] for p in eval_points], marker="o", color="orange")
axes[1].set_title("Val macro F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## Evaluation

In [ ]:
pred_output = trainer.predict(test_hf)
test_logits = pred_output.predictions
test_labels = pred_output.label_ids
test_preds = np.argmax(test_logits, axis=-1)

test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")
test_metrics = {"macro_f1": test_macro_f1, "weighted_f1": test_weighted_f1}

print(f"test macro f1: {test_macro_f1:.4f}")
print(f"test weighted f1: {test_weighted_f1:.4f}")

report = classification_report(
    test_labels,
    test_preds,
    target_names=[id2label[i] for i in range(NUM_LABELS)],
    output_dict=True,
    zero_division=0,
 )
report_df = pd.DataFrame(report).T
display(report_df[["precision", "recall", "f1-score", "support"]])

metrics_path = os.path.join(RESULTS_DIR, "test_metrics.json")
report_path = os.path.join(RESULTS_DIR, "classification_report.csv")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)
report_df.to_csv(report_path, index=True)

## Checkpoint Saving

In [ ]:
os.makedirs(SAVE_PATH, exist_ok=True)
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

reloaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
reloaded_model.to(DEVICE)
reloaded_model.eval()

inputs = reloaded_tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LEN,
 )
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.no_grad():
    logits = reloaded_model(**inputs).logits
pred_id = int(torch.argmax(logits, dim=-1).item())

print("sample text:", sample_text)
print("pred label:", id2label[pred_id])
print("gold label:", sample_gold_label)

## Summary
- Model: `MODEL_NAME`
- Test metrics: macro F1 in `test_metrics["macro_f1"]`, weighted F1 in `test_metrics["weighted_f1"]`
- Limitations: 20k sample cap, no hyperparameter search, class imbalance